# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [7]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [8]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [10]:
# 🤖 AGENT FUNCTION

import re
import logging
import warnings

# Configure basic logging for observability (bonus requirement)
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


def agent(query: str) -> dict:
    """
    Single-agent routing pipeline.

    Routing rules (checked in order):
      1. 'calculate' in query  →  Calculator Tool
      
      2. 'keywords' in query   →  Keyword Extractor Tool
      3. Non-empty string      →  General fallback response
      4. Anything else         →  Error response
    """
    try:
        # Guard: reject empty / non-string input early
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query must be a non-empty string.")

        query_lower = query.lower()
        logger.info("Received query: %r", query)

        # ── Route 1: Math calculation ────────────────────────────────────
        if "calculate" in query_lower:
            # Extract everything after the word 'calculate'
            match = re.search(r'calculate\s+(.+)', query, re.IGNORECASE)
            if not match:
                raise ValueError("No expression found after 'calculate'.")

            expression = match.group(1).strip()
            logger.info("Routing to calculator with expression: %r", expression)

            result = calculator(expression)

            # calculator() returns a string; surface explicit errors
            if result == "Error in calculation":
                return {"type": "error", "result": f"Could not evaluate expression: '{expression}'"}

            return {"type": "calculation", "result": result}

        # ── Route 2: Keyword extraction ──────────────────────────────────
        elif "keywords" in query_lower:
            # Accept both 'keywords from <text>' and 'keywords: <text>'
            match = re.search(r'keywords(?:\s+from|:)?\s+(.+)', query, re.IGNORECASE)
            if not match:
                raise ValueError("No text found to extract keywords from.")

            text = match.group(1).strip()
            logger.info("Routing to keyword extractor with text: %r", text)

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # ── Route 3: General conversational fallback ─────────────────────
        else:
            logger.info("Routing to general fallback.")
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. "
                          "Please ask me to 'calculate <expression>' or "
                          "'extract keywords from <text>' for tool-assisted responses."
            }

    except ValueError as ve:
        # Malformed / unroutable command
        logger.warning("ValueError: %s", ve)
        return {"type": "error", "result": str(ve)}
    except Exception as exc:
        # Catch-all for unexpected failures
        logger.error("Unexpected error: %s", exc)
        return {"type": "error", "result": f"Unexpected error: {exc}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [11]:
# 🧪 Test Cases (8 queries covering every routing path)

queries = [
    # Calculation – valid
    "Calculate 20 + 5",
    # Keywords – valid
    "Extract keywords from Artificial Intelligence is transforming industries"
    # General fallback
    "What is machine learning?"
]

for q in queries:
    print("Query  :", repr(q))
    print("Response:", agent(q))
    print("-" * 50)


[INFO] Received query: 'Calculate 20 + 5'
[INFO] Routing to calculator with expression: '20 + 5'
[INFO] Received query: 'Extract keywords from Artificial Intelligence is transforming industriesWhat is machine learning?'
[INFO] Routing to keyword extractor with text: 'Artificial Intelligence is transforming industriesWhat is machine learning?'


Query  : 'Calculate 20 + 5'
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query  : 'Extract keywords from Artificial Intelligence is transforming industriesWhat is machine learning?'
Response: {'type': 'keywords', 'result': ['artificial', 'machine', 'transforming', 'industrieswhat', 'intelligence']}
--------------------------------------------------


In [12]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop):  Calculate 3+5


[INFO] Received query: 'Calculate 3+5'
[INFO] Routing to calculator with expression: '3+5'


Response: {'type': 'calculation', 'result': '8'}


Enter query (type 'exit' to stop):  Extract keywords from Artificial Intelligence is transforming industries


[INFO] Received query: 'Extract keywords from Artificial Intelligence is transforming industries'
[INFO] Routing to keyword extractor with text: 'Artificial Intelligence is transforming industries'


Response: {'type': 'keywords', 'result': ['industries', 'transforming', 'intelligence', 'artificial']}


Enter query (type 'exit' to stop):  What is machine learning?


[INFO] Received query: 'What is machine learning?'
[INFO] Routing to general fallback.


Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. Please ask me to 'calculate <expression>' or 'extract keywords from <text>' for tool-assisted responses."}


Enter query (type 'exit' to stop):  exit
